# 01 — Data & Quality

Loads QSCI's real source datasets and runs `qsci_core.data.quality` checks against them. Every result in this notebook is **LIVE** — computed from real data, right now — or explicitly marked otherwise if a dataset isn't available yet.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from qsci_core.data import loaders, quality

RAW_DIR = PROJECT_ROOT / "data" / "raw"
print("project root:", PROJECT_ROOT)
print("raw data dir:", RAW_DIR)


project root: D:\Projects\QML project
raw data dir: D:\Projects\QML project\data\raw


## CVRPLIB — routing ground truth

Status: **LIVE**. Small Augerat-family instances downloaded directly from the CVRPLIB mirror (`galgos.inf.puc-rio.br/cvrplib`), parsed with `vrplib`.

In [2]:
CVRPLIB_INSTANCES = ["A-n32-k5", "A-n33-k5", "A-n37-k5", "B-n31-k5"]

cvrplib_reports = {}
for name in CVRPLIB_INSTANCES:
    instance, solution = loaders.load_cvrplib_instance(name)
    report = quality.assess_cvrplib_instance(instance, name=name)
    cvrplib_reports[name] = (report, solution)
    print(report.to_markdown())
    if solution is not None:
        print(f"- known best-known cost: {solution.get('cost')}")
    print()


### Data Quality Report — CVRPLIB A-n32-k5 [LIVE]
- rows: 32  cols: 2
- duplicate rows: 0
- negative values: 
- known best-known cost: 784

### Data Quality Report — CVRPLIB A-n33-k5 [LIVE]
- rows: 33  cols: 2
- duplicate rows: 0
- negative values: 
- known best-known cost: 661

### Data Quality Report — CVRPLIB A-n37-k5 [LIVE]
- rows: 37  cols: 2
- duplicate rows: 0
- negative values: 
- known best-known cost: 669

### Data Quality Report — CVRPLIB B-n31-k5 [LIVE]
- rows: 31  cols: 2
- duplicate rows: 2
- negative values: 
- known best-known cost: 672



## M5 Forecasting — Accuracy

Status: depends on whether `data/raw/m5/` has been populated via `kaggle competitions download -c m5-forecasting-accuracy`. If the files aren't present yet, this section reports **PENDING** rather than fabricating a result.

In [3]:
m5_dir = RAW_DIR / "m5"
m5_files_present = m5_dir.exists() and any(m5_dir.glob("sales_train_*.csv"))

if not m5_files_present:
    print("STATUS: PENDING — M5 files not found under", m5_dir)
    print("Run: kaggle competitions download -c m5-forecasting-accuracy -p", m5_dir)
    m5_report = None
else:
    m5 = loaders.load_m5(m5_dir)
    m5_report = quality.assess_m5(m5["sales"], m5["calendar"], m5["prices"])
    print(m5_report.to_markdown())


### Data Quality Report — M5 Forecasting - Accuracy (sales_train) [LIVE]
- rows: 30,490  cols: 1947
- duplicate rows: 0
- date range: 2011-01-29 to 2016-06-19 (gaps: 0)
- negative values: 
- zero-inflation %: unit_sales (all d_ cols)=68.00%
- IQR outlier %: sell_price=4.44%
- notes: sales has 1941 day columns but calendar has 1969 rows (expected: evaluation file covers more days than validation)


## Amazon Last-Mile Routing Research Challenge

Status: depends on whether `data/raw/last_mile/` has been synced via `aws s3 sync --no-sign-request s3://amazon-last-mile-challenges/almrrc2021/ data/raw/last_mile/`. This sync is ~3.1GB and may still be in progress — if the JSON files aren't present, this section reports **PENDING**.

In [4]:
import json

last_mile_dir = RAW_DIR / "last_mile"
route_files = list(last_mile_dir.rglob("route_data.json")) if last_mile_dir.exists() else []

if not route_files:
    print("STATUS: PENDING — Last-Mile route_data.json not found under", last_mile_dir)
    last_mile_report = None
else:
    last_mile = loaders.load_last_mile(last_mile_dir)
    route_data = last_mile["route_data"]
    n_routes = len(route_data)
    n_stops = sum(len(r.get("stops", {})) for r in route_data.values())
    print(f"STATUS: LIVE — {n_routes:,} routes, {n_stops:,} total stops loaded from real Amazon Last-Mile data")
    print("keys loaded:", list(last_mile.keys()))


STATUS: LIVE — 6,112 routes, 904,527 total stops loaded from real Amazon Last-Mile data
keys loaded: ['route_data', 'package_data', 'travel_times', 'actual_sequences', 'invalid_sequence_scores']
